MSBDS 566 - Assingnment 1
Sheretta T. Barnes
MSBDS 566
September 17, 2026
Honor Statement: I affirm that I have completed this assignment in accordance with the course academic integrity requirements.

**Part A. "In your own words (3–5 sentences), restate the business problem behind this dataset: who would act on the prediction, what decision it supports, and what ""success"" means in measurable terms, the same shape as the Business Understanding slide from Week 2.**"


1. The business problem behind the **diabetes hospital-readmission dataset** is to identify patients who are high-risk of a 30-day readmission so care teams of professionals can intervene before discharge. The care team would be the decision makers to determine who is at risk of readmission. The success would be to reduce 30-day readmissions by 20% without creating an increase in the number of false alerts.

**Then, in 1–2 sentences, describe ONE way you would adapt this problem for a different but related question (e.g., a different age group, a different readmission window, a different type of intervention). You do not need to build
this variant — just describe it clearly enough that someone else could.**

1. One way to adapt this problem would be to predict which patients with diabetes are at risk of readmission within 60 days rather than 30 days. This would allow care teams to provide longer-term follow-up support after discharge.



**Part B — Analyze Two Data Decisions
(3 points — SO 1.1: analyze a data science solution)
Choose TWO decisions from our Week 2–3 pipeline (for example: the missing-data strategy, the ICD-9 diagnosis grouping, the insulin ordinal/nominal choice, or the patient-grouped train/test split).**

1. I am chosing the missing data strategy that we chose, which involved how to hand the approximately 2% of observations with missing race information. Rather than deleting these observations or imputing a racial category, we retained them and created a separate "missing" category, which was then one-hot encoded with the other race categories. I believe overall that there are multiple ways to handle missing data. The way we chose to handle is to drop some of the missing data. We could either replace the missing values through various methods (imputation, etc.) or we could drop the missing values. There are some alternatives, but in my experience this is contingent on first determining who is MCAR, MNAR, and MAR. It would also depend on the type of data we are replacing or deleting. Trained as a social scientist, there are real consequences in how we handle the data. However, the decision we chose would hurt model fainess if the "missing" category combines patients whose race was unrecorded for very different reasons. For exampe, if certian racial or ethnic groups are disproportionately represented among patients with race information, the model could learn that missingingess predicts readmission without identifying the healthcare or documentation practices causing that patterns. This could lead to less equitable predictions for those patients. An alternative approach could have been to move forward with the missinginess and do some sensitivity analyses to determine if there are signficant differences between missing vs. non-missingness. For instance, I would expect some amounts of missing data on clincial or hospital variables because that is contingent upon the patients health history, so using missing values or dropping could be harmful in this context and misleading in the findings. Substituing on demogrpahics variables (race, gender, income) can also be problematic because of biases about certain populations. So in this case, I think there are multiple ways one can address patterns of missingingness but the goal to is to be as ethical as possible and be sure we are referring the codebook, searhing the literature to determine how others have addressed this, and using the appropriate statistical techniques.

2. A second decision was to divide the training and testing data by patient rather than randomly splitting hospital encounters. The alternative would have been a standard random split in which the different encounters from the same patient could appear in the both the training and testing sets. We chose a patient-grouped split to prevent data leakage becasue the encounters from the same patient may contain similar diagnoses, treatment histories, and demographic characteristics. If the same patient appeared in both sets the model might seem more accurate because it had already learned information associated with that person. However, a group split could also hurt performance if a small number of patients with repeated admissions or characteristics were in only one dataet. For example, if most of the patients with multiple readmissions were placed in the test set, the training set might not contain enough comparable cases for the model to learn their patterns, leading to poorer readmission predictions for high-risk patients.

**Part C — Design Memo
(2 points — SO 2.1: design a data science solution)
Before you write any modeling code, write a short design memo (this is Stage 3→4 of CRISP-DM — planning
before building):
• Which TWO model families will you compare? (e.g., logistic regression and one other model family covered
in class)
• Which evaluation metric(s) will you report, and why — tie this to the class-balance discussion from Week 2
(why not accuracy alone?)
• One risk you will watch for (e.g., leakage, overfitting, class imbalance) and how your plan guards against it.**

1. I would compare logistic regression and random forest models to predict whether a patient will be readmitted within 30 days. The logistic regression will provide me with an baseline model and show how each of the features in my model is associated with the probability of readmission. The random forest technique will allow me to examine whether a tree-based model that captures complex relationships and interactions produces better predictions. Consistent with what is typically reported I would report the following: accuracy, precision, recall, F1-score, and the ROC-AUC. One thing to note is that as we learned in this class and previous classes that accuracy alone could be misleading because the outcome classes may be imbalanced. For instance, if fewer patients were readmitted than not readmitted, a model could achieve high accuracy simply by predicting that most pateints will be be readmitted. Recall will tell us how well the models identify patients who are actually readmitted, precision will tell us how often predicted readmissions are correct, and the F1 score will tell us about the balance precision and recall. ROC-AUC will measure each model's abililty to distinguish between patients who are and are not readmitted across the classification thresholds. In terms of risk, one of the risk I will monitor is overfitting, especially for the random forest model. A random forest model may perform well on the training data but poorly on new patients. I will guard against this by training the models only on the training data, tuning the random forest's hyperparameters using cross-validation, and making the final comparion on the untouched test data. I will also use the patient-group train/test split to prevent encounters from the same patient from appearing in both datasets and causing data leakage.



**Part D — Implement Two Baseline Models
(7 points — SO 2.2: implement a data science solution)
Load your saved train/test files and implement the two models you planned in Part C. Your notebook must:
• Run top to bottom without errors (Kernel → Restart & Run All before you submit)
• Use the SAME train/test split your Data Preparation notebook created — do not re-split the data
• Include clear section headers and comments explaining each step, in the style of our class notebooks
• Print each model's predictions on the test set (a small sample is fine)**


In [4]:
import pandas as pd

# Load the model-ready diabetes files
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")

X_train_scaled = pd.read_csv("X_train_scaled.csv")
X_test_scaled = pd.read_csv("X_test_scaled.csv")

y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

# Confirm that the files loaded correctly
print("Files loaded successfully.")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Files loaded successfully.
X_train: (79567, 33)
X_test: (19773, 33)
X_train_scaled: (79567, 33)
X_test_scaled: (19773, 33)
y_train: (79567,)
y_test: (19773,)


**Logistic Model Logistic is the baseline model**

I am using this model to predict the 30-day readmission. The scaled predictor files are used because logistic regression can be affected by the differences in the variable scales.

In [5]:
# Import logistic regression
from sklearn.linear_model import LogisticRegression

# Create the baseline logistic regression model
log_model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model using the scaled training data
log_model.fit(X_train_scaled, y_train)

# Generate predictions and probabilities for the test set
log_predictions = log_model.predict(X_test_scaled)
log_probabilities = log_model.predict_proba(X_test_scaled)[:, 1]

# Display a small sample of predictions
log_results = pd.DataFrame({
    "Actual": y_test.iloc[:10].values,
    "Predicted": log_predictions[:10],
    "Probability_Readmitted": log_probabilities[:10]
})

print("Logistic Regression — First 10 Test Predictions")
display(log_results)

Logistic Regression — First 10 Test Predictions


,Actual,Predicted,Probability_Readmitted
0,0,0,0.097978
1,0,0,0.066892
2,0,0,0.092302
3,0,0,0.081493
4,0,0,0.065892
5,0,0,0.093824
6,0,0,0.073493
7,0,0,0.092543
8,0,0,0.117796
9,0,0,0.083239


**Random Forest is the second model I am using. It caputres the nonlinear relationships and interaction among the predictors. The unscaled predictor Files are used because tree-based models do not require standarized variables.**


In [6]:

# Import random forest
from sklearn.ensemble import RandomForestClassifier

# Create the baseline random forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model using the unscaled training data
rf_model.fit(X_train, y_train)

# Generate predictions and probabilities for the test set
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

# Display a small sample of predictions
rf_results = pd.DataFrame({
    "Actual": y_test.iloc[:10].values,
    "Predicted": rf_predictions[:10],
    "Probability_Readmitted": rf_probabilities[:10]
})

print("Random Forest — First 10 Test Predictions")
display(rf_results)

Random Forest — First 10 Test Predictions


,Actual,Predicted,Probability_Readmitted
0,0,0,0.01
1,0,0,0.02
2,0,0,0.14
3,0,0,0.08
4,0,0,0.07
5,0,0,0.06
6,0,0,0.09
7,0,0,0.09
8,0,0,0.13
9,0,0,0.10


**Overall, the preliminary results suggest that both models predicted zero for the first 10 observations. For logistic regression, the estimated probabilities of readmission ranged from approximately 6.6% to 11.18%. For random forest, the estimated probabilities ranged from 1% to 14%. It is important to note that these 10 observations are only a small sample and can't estabilish which model performed better, model performance much be evaluated.**

**The next step in the process is to evaluate and compare.**

**Part E**

In [7]:
# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Function to calculate model performance
def evaluate_model(model_name, actual, predicted, probabilities):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1_Score": f1_score(actual, predicted, zero_division=0),
        "ROC_AUC": roc_auc_score(actual, probabilities)
    }

# Evaluate logistic regression
log_metrics = evaluate_model(
    "Logistic Regression",
    y_test,
    log_predictions,
    log_probabilities
)

# Evaluate random forest
rf_metrics = evaluate_model(
    "Random Forest",
    y_test,
    rf_predictions,
    rf_probabilities
)

# Create a comparison table
comparison_table = pd.DataFrame([log_metrics, rf_metrics])

# Display results rounded to three decimal places
display(comparison_table.round(3))

,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,Logistic Regression,0.887,0.688,0.010,0.019,0.641
1,Random Forest,0.886,0.380,0.012,0.023,0.593


**Overall, the results revealed that logistic regression performed better.**

In this comparison, I used logistic regression and random forest. The logistic regression and random forest produced similar accuracy values of .887 and .886, respectively. However, the accuracy is misleading in this dataset becasue 89% of parents were not readmitted within 30 days. A model would therefore achieve high accuracy primarily by predicting the majority outcome of no readmission. Logistic regression also had higher precision than random forest (.68 in comparsion to .38) and a higher ROC-AUC (.641 to .593) indicating that logistic regression was better at distinguishing between patients who were and were not admitted across classification thresholds. Random forest produced slightly higher recall (.012 in comparion to .010). and a F1-socre of (0.023 verus 0.019).

Overall, I would recommend logistic regression as the stronger of the two models because it had better precision and ROC-AUC, accuracy, and was easier to interpret. This recommendaion is consisent with the Part C plan to evaluate the models using multiple metrics rather than accuracy alone.

One surprising finding is that random forest did not outperform logistic regression. In previous classes, this is sort of an assumption that random forest is superior to logistic regression. Instead it had a lower ROC-AUC and lower precision. Another important findng was that both models achieved nearly 89% accuracy while detecting almost none of the patients who were actually readmitted. This suggest that accuracy alone is not an appropriate measure of performance for this imbalanced outcome.

**Part F — Reflection
(1 point — SO 3.2: technical writing skills in lab reports and projects)
In 3–5 sentences: what would you do differently with more time or more data, and what is one thing that surprised
you while working through this assignment**

**One thing that surprised me was that the logistic regression performed better than random forest, even though logistic regression is often treated as a baseline model before moving to more complex approaches. This result taught me that a more complex model is not necessarily a better model and that model selection should be deliberate and based on how well each approach fits the research problem and data. With more time, I would test additional approaches for addressing class imbalance, tune the models, and examine how other models or classification thresholds improve the identification of patients who are readmitted. Overall, this assignment reinforced the importance of understanding what each model does and why it is being included in a comparison.**